# Session 9 Programming for Maqueen Robot

**Learning goals:**

- Set up a MicroPython project in the micro:bit Python Editor and include the `maqueen.py` driver library
- Initialise the Maqueen robot car and navigate its hardware layout, pin connections, and I2C motor controller
- Control on-board indicator LEDs and addressable NeoPixel RGB LEDs with custom colours and patterns
- Drive and steer dual DC motors with forward, reverse, pivot turn, and spin manoeuvres
- Measure distance using the HC-SR04 ultrasonic sensor and implement reactive obstacle avoidance
- Read infrared line-tracking patrol sensors to detect dark lines on light surfaces
- Apply lists and dictionaries to build data-driven robot state machines and autonomous mission plans
- Use the 5x5 LED display, NeoPixels, headlights, and buzzer for untethered on-board feedback

**Prerequisites:**

- Python variables, data types, and operators (Session 2)
- Conditional statements (`if`/`elif`/`else`) and Boolean logic (Session 3)
- `while` loops, timing delays, and function design (Sessions 4–6)
- Core Python data structures: lists, tuples, and dictionaries (Session 7–8)
- BBC micro:bit board with DFRobot Micro:Maqueen chassis and battery power

> **Note on Running Code and On-Board Feedback:** The code cells in this session are written in **MicroPython for the BBC micro:bit**. When your robot runs untethered across the floor on battery power, reading `print()` text on a computer screen is inconvenient. Instead, we use the micro:bit's **5x5 LED display** (`display.show()`), **headlight indicators** (`robot.led_left/right`), **NeoPixel RGB underglow**, and **buzzer notes** (`music.pitch()`) for live status feedback.

## Outline {.unnumbered #sec-outline}

1. **Setting up the environment**: using the official micro:bit Python Editor and uploading `maqueen.py`
2. **Hardware and driver architecture**: pin mappings, I2C motor control, and class methods
3. **Initialising the robot**: creating a `Maqueen` instance and providing on-board startup feedback
4. **Indicator LEDs and display cues**: signalling status with headlights and 5x5 LED images
5. **NeoPixel RGB lighting**: setting colours and cycling patterns with lists and tuples
6. **Motor control and movement**: driving forwards, reversing, pivoting, and spinning
7. **Ultrasonic distance measurement**: measuring obstacle proximity with live display warnings
8. **Infrared line-following sensors**: reading the patrol sensors to detect tracks
9. **Practical robotics projects**: state machine dictionaries, lookup tables, and list-driven mission routes
10. **Exercises and solutions**: structured coding challenges with reference solutions
11. **Useful resources**: links to official documentation, the online editor, and hardware guides

## 1. Setting Up the Environment {#sec-environment}

Programming physical robotics requires connecting software instructions to hardware peripherals. We use the official **micro:bit Python Editor** at [https://python.microbit.org/v/3](https://python.microbit.org/v/3), which runs entirely in your web browser.

### Setup Steps

1. Open [https://python.microbit.org/v/3](https://python.microbit.org/v/3) in Google Chrome, Microsoft Edge, or another WebUSB-compatible browser.
2. In the editor's left-hand sidebar, select **Project** (the folder icon).
3. Select **Open** or **Add file** and upload the `maqueen.py` driver library file into your project.
4. When `maqueen.py` appears in the project file explorer, it will automatically bundle into the `.hex` binary file whenever you send code to the micro:bit.
5. Write your robot control logic in `main.py`.
6. Connect your BBC micro:bit to your computer via a micro-USB cable and click **Send to micro:bit** to flash the program.

> **Why do we use `maqueen.py`?**
>
> The Maqueen chassis communicates with the micro:bit using both direct GPIO pins and an on-board I2C microcontroller (at I2C address `0x10`). The `maqueen.py` driver library abstracts low-level I2C register writes and pulse timing into intuitive Python methods so you can focus on writing high-level robotics algorithms.

## 2. Hardware and Driver Architecture {#sec-architecture}

The DFRobot Micro:Maqueen robot integrates multiple sensors, actuators, and power management circuits onto a single mobile chassis.

### Pin and Bus Mapping

The table below outlines how the micro:bit edge connector interacts with the Maqueen hardware:

| Hardware Component | micro:bit Interface | Description |
|:---|:---|:---|
| Dual DC Motors | I2C Bus (`0x10`) | Controlled via I2C commands sent to the chassis motor controller |
| Ultrasonic Sensor Trigger | `pin1` | Digital output trigger pulse for HC-SR04 sonar |
| Ultrasonic Sensor Echo | `pin2` | Digital input timing pulse for HC-SR04 sonar |
| Left Indicator LED | `pin8` | Digital output for the front-left red LED |
| Right Indicator LED | `pin12` | Digital output for the front-right red LED |
| Left Line Patrol Sensor | `pin13` | Digital input for the underside left infrared phototransistor |
| Right Line Patrol Sensor | `pin14` | Digital input for the underside right infrared phototransistor |
| NeoPixel RGB LEDs (×4) | `pin15` | Single-wire digital control line for the 4 addressable RGB LEDs |
| On-board Buzzer | `pin0` / Audio | micro:bit sound output for audio feedback |

### The `Maqueen` Driver Library API

The `maqueen.py` library provides a single `Maqueen` class with convenient methods grouped into functional areas:

| Category | Method | Description |
|:---|:---|:---|
| **Indicator LEDs** | `led_left(value)` / `led_right(value)` | Turn left/right red indicator LEDs on (`1`) or off (`0`) |
| **NeoPixel RGB** | `rgb_front_left(r, g, b)`, `rgb_front_right(r, g, b)`, `rgb_rear_left(r, g, b)`, `rgb_rear_right(r, g, b)` | Set individual RGB LED colours (`0`–`255` per channel) |
| **Motors** | `set_motor(motor, value)` | Drive motor `0` (left) or `1` (right) at speed `-255` to `255` |
| **Motors** | `motor_stop_all()` | Stop both motors immediately |
| **Ultrasonic** | `read_distance()` / `ultrasound_measure()` | Read front obstacle distance in centimetres |
| **Line Patrol** | `line_left()` / `line_right()` / `read_patrol(sensor)` | Read line sensor state (`1` = dark line, `0` = light surface) |

## 3. Initialising the Robot {#sec-initialisation}

Every robot program begins by importing the required MicroPython libraries and creating an instance of the `Maqueen` class. The constructor automatically configures the NeoPixel strip on `pin15` and sets the ultrasonic trigger pin to a ready state.

The initialisation test script below creates the robot object, displays a ready icon on the 5x5 LED screen, flashes green NeoPixels, and plays a startup chime.

In [ ]:
# Import core micro:bit hardware module and Maqueen driver
from microbit import *
from maqueen import Maqueen
import utime
import music

# Initialise the Maqueen robot instance
robot = Maqueen()

# Visual feedback: show smiling face on 5x5 LED matrix
display.show(Image.HAPPY)

# Visual feedback: illuminate green underglow NeoPixels
robot.rgb_front_left(0, 255, 0)
robot.rgb_front_right(0, 255, 0)

# Auditory feedback: play ascending startup melody
music.play(music.POWER_UP)

# Hold startup display for 1 second, then clear
utime.sleep(1.0)
display.show(Image.HEART)

**Expected on-board behaviour:**

- The 5x5 LED display transitions from `HAPPY` to `HEART`.
- Both front NeoPixel LEDs glow green.
- The micro:bit plays the ascending `POWER_UP` chime.
- If no lights or sounds activate, verify that the chassis power switch is in the **ON** position.

## 4. Indicator LEDs {#sec-indicator-leds}

The Maqueen chassis features two front-facing red LEDs connected to digital outputs `pin8` and `pin12`. These are ideal for indicator signals, state alerts, and hazard warnings.

| Method | Parameter | Effect |
|:---|:---|:---|
| `robot.led_left(value)` | `1` = on, `0` = off | Controls the front-left red indicator LED (`pin8`) |
| `robot.led_right(value)` | `1` = on, `0` = off | Controls the front-right red indicator LED (`pin12`) |

### Example — Alternating Hazard Flasher with Display Sync

This program runs a loop that synchronises the front red headlights with direction arrows on the 5x5 matrix.

In [ ]:
# Alternating indicator LED pattern with 5x5 display sync
from microbit import *
from maqueen import Maqueen
import utime

# Initialise robot driver
robot = Maqueen()

# Define timing constant in seconds
FLASH_INTERVAL_SECONDS = 0.5

# Run alternating flasher sequence
for _ in range(6):
    # Left indicator on: show West arrow on LED screen
    display.show(Image.ARROW_W)
    robot.led_left(1)   # value: 1 (on)
    robot.led_right(0)  # value: 0 (off)
    utime.sleep(FLASH_INTERVAL_SECONDS)

    # Right indicator on: show East arrow on LED screen
    display.show(Image.ARROW_E)
    robot.led_left(0)   # value: 0 (off)
    robot.led_right(1)  # value: 1 (on)
    utime.sleep(FLASH_INTERVAL_SECONDS)

# Turn off both LEDs and clear screen at end of sequence
robot.led_left(0)
robot.led_right(0)
display.clear()

## 5. NeoPixel RGB Lighting {#sec-neopixels}

The Maqueen chassis is equipped with **four addressable WS2812B NeoPixel RGB LEDs** mounted under the chassis corners to illuminate the ground.

### LED Positioning and Methods

Each NeoPixel is addressed through its corresponding helper method:

| Position | Method | Index on Strip |
|:---|:---|:---:|n| Front-Left | `robot.rgb_front_left(r, g, b)` | 0 |
| Rear-Left | `robot.rgb_rear_left(r, g, b)` | 1 |
| Rear-Right | `robot.rgb_rear_right(r, g, b)` | 2 |
| Front-Right | `robot.rgb_front_right(r, g, b)` | 3 |

Each colour parameter (`r`, `g`, `b`) accepts an integer between `0` (off) and `255` (full brightness).

### Example — Cycling a List of Colour Tuples

Using a **list of colour tuples** from Session 7–8, this program iterates through four underglow shades while showing corresponding letters on the 5x5 display.

In [ ]:
# NeoPixel RGB colour cycling using a list of colour tuples
from microbit import *
from maqueen import Maqueen
import utime

# Initialise robot instance
robot = Maqueen()

# Define list of colour dictionaries with label, RGB tuple, and display character
COLOUR_PALETTE = [
    {"label": "R", "rgb": (255, 0, 0)},     # Red
    {"label": "G", "rgb": (0, 255, 0)},     # Green
    {"label": "B", "rgb": (0, 0, 255)},     # Blue
    {"label": "Y", "rgb": (255, 200, 0)},   # Yellow
]

STEP_DELAY_SECONDS = 0.6

# Iterate through colour list
for colour_info in COLOUR_PALETTE:
    # Display initial letter on screen (e.g. 'R', 'G', 'B', 'Y')
    display.show(colour_info["label"])

    # Unpack RGB tuple
    red_val, green_val, blue_val = colour_info["rgb"]

    # Apply colour to all 4 under-chassis NeoPixels
    robot.rgb_front_left(red_val, green_val, blue_val)
    robot.rgb_front_right(red_val, green_val, blue_val)
    robot.rgb_rear_left(red_val, green_val, blue_val)
    robot.rgb_rear_right(red_val, green_val, blue_val)

    utime.sleep(STEP_DELAY_SECONDS)

# Turn off all RGB LEDs and clear screen
robot.rgb_front_left(0, 0, 0)
robot.rgb_front_right(0, 0, 0)
robot.rgb_rear_left(0, 0, 0)
robot.rgb_rear_right(0, 0, 0)
display.clear()

## 6. Motor Control and Movement {#sec-motors}

The Maqueen uses two miniature DC gear motors connected to rubber wheels. A dedicated motor driver microcontroller on the chassis receives movement commands from the micro:bit over the I2C serial bus.

### The `set_motor()` Method

```python
robot.set_motor(motor, value)
```

- **`motor`**: `0` for the left motor, `1` for the right motor.
- **`value`**: Speed integer from **`-255`** (full reverse) to **`255`** (full forward). A value of `0` stops the motor.

To stop both motors simultaneously, call `robot.motor_stop_all()`.

### Steering Manoeuvres

By varying the speed and direction of the left and right motors relative to each other, you can achieve different movement patterns:

| Movement Type | Left Motor (`motor=0`) | Right Motor (`motor=1`) | Description |
|:---|:---:|:---:|:---|
| **Forward** | Positive (e.g. `100`) | Positive (e.g. `100`) | Both wheels drive forward at equal speed |
| **Reverse** | Negative (e.g. `-100`) | Negative (e.g. `-100`) | Both wheels drive backward at equal speed |
| **Pivot Turn Right** | Forward (e.g. `120`) | Stopped (`0`) | Left wheel drives while right wheel acts as a stationary pivot |
| **Pivot Turn Left** | Stopped (`0`) | Forward (e.g. `120`) | Right wheel drives while left wheel acts as a stationary pivot |
| **Spin Turn (Spot)** | Forward (`100`) | Reverse (`-100`) | Wheels turn in opposite directions to rotate on the spot |

### Example — Drive Forward, Pivot, and Spin

In [ ]:
# Motor control and steering manoeuvre demonstration with screen feedback
from microbit import *
from maqueen import Maqueen
import utime

# Initialise robot
robot = Maqueen()

# Define movement speed constants
SPEED_CRUISE = 100   # moderate forward speed (0 to 255)
SPEED_PIVOT  = 120   # slightly higher torque for single-wheel pivot
SPEED_STOP   = 0     # stationary

# Step 1: Drive forward for 1.5 seconds
display.show(Image.ARROW_N)
robot.set_motor(0, SPEED_CRUISE)  # left motor forward at speed 100
robot.set_motor(1, SPEED_CRUISE)  # right motor forward at speed 100
utime.sleep(1.5)                  # duration: drive forward for 1.5s

# Step 2: Stop briefly to prevent momentum overshoot
robot.motor_stop_all()
display.show(Image.DIAMOND_SMALL)
utime.sleep(0.3)

# Step 3: Pivot turn right for 0.8 seconds
display.show(Image.ARROW_E)
robot.set_motor(0, SPEED_PIVOT)   # left motor drives forward
robot.set_motor(1, SPEED_STOP)    # right motor stationary
utime.sleep(0.8)

robot.motor_stop_all()
utime.sleep(0.3)

# Step 4: Spin turn on the spot for 0.6 seconds
display.show(Image.ALL_CLOCKS[0])
robot.set_motor(0, SPEED_CRUISE)   # left motor forward
robot.set_motor(1, -SPEED_CRUISE)  # right motor reverse
utime.sleep(0.6)

# Step 5: Final full stop
robot.motor_stop_all()
display.show(Image.HAPPY)

## 7. Ultrasonic Distance Measurement {#sec-ultrasonic}

The Maqueen includes an **HC-SR04** ultrasonic range finder mounted at the front. The sensor emits high-frequency sound pulses from its transmitter transducer (`pin1` trigger) and times the return echo on its receiver transducer (`pin2` echo).

### Measurement Methods

1. **`robot.read_distance()` (Recommended)**:
   - Uses MicroPython's high-resolution `machine.time_pulse_us()` internally.
   - **Returns**: `float` representing distance in centimetres.
   - **Reliable Range**: approximately 2 cm to 250 cm.
2. **`robot.ultrasound_measure()`**:
   - Software-timed pulse measurement.
   - **Returns**: `int` distance in centimetres, or error codes (`-1` = timeout waiting for echo, `-2` = echo pulse duration invalid).

### Example — Distance Proximity Monitor with Display Feedback

The program below checks proximity in real time: it displays a smiling face with green NeoPixels when the path is clear, and switches to a skull icon with flashing red headlights and a warning tone when an obstacle is within 15 cm.

In [ ]:
# Ultrasonic distance sensing with on-board audiovisual alerts
from microbit import *
from maqueen import Maqueen
import utime
import music

# Initialise robot driver
robot = Maqueen()

# Define safety threshold in centimetres
PROXIMITY_THRESHOLD_CM = 15.0

# Collect 20 consecutive distance samples
for _ in range(20):
    distance_cm = robot.read_distance()  # returns distance in cm as float

    if distance_cm < PROXIMITY_THRESHOLD_CM and distance_cm > 0:
        # Obstacle detected: show skull icon, red NeoPixels, headlights on, warning tone
        display.show(Image.SKULL)
        robot.led_left(1)
        robot.led_right(1)
        robot.rgb_front_left(255, 0, 0)
        robot.rgb_front_right(255, 0, 0)
        music.pitch(880, 100)  # 880 Hz tone for 100 ms
    else:
        # Path clear: show happy face, green NeoPixels, headlights off
        display.show(Image.HAPPY)
        robot.led_left(0)
        robot.led_right(0)
        robot.rgb_front_left(0, 255, 0)
        robot.rgb_front_right(0, 255, 0)

    utime.sleep(0.3)  # duration: wait 300 ms between samples

# Clear indicators at end of test
robot.led_left(0)
robot.led_right(0)
robot.rgb_front_left(0, 0, 0)
robot.rgb_front_right(0, 0, 0)
display.clear()

## 8. Infrared Line Sensors {#sec-line-sensors}

Underneath the front chassis, the Maqueen carries two **infrared (IR) phototransistor line patrol sensors**. Each sensor contains an IR emitter LED and a photodetector.

- When positioned over a **light or reflective surface**, the infrared beam bounces back into the detector, resulting in a reading of `0`.
- When positioned over a **dark, non-reflective surface** (such as black electrical tape), the infrared light is absorbed, resulting in a reading of `1`.

### Line Sensor Methods

| Method | micro:bit Pin | Return Value | Description |
|:---|:---:|:---:|:---|
| `robot.line_left()` | `pin13` | `1` or `0` | Returns `1` when the left sensor detects a dark line |
| `robot.line_right()` | `pin14` | `1` or `0` | Returns `1` when the right sensor detects a dark line |
| `robot.read_patrol(sensor)` | `pin13` / `pin14` | `1` or `0` | Pass `0` for left sensor, `1` for right sensor |

### Example — Live Patrol Sensor Tester with On-Board Display

This program displays which sensors detect the line directly on the 5x5 LED matrix using directional arrows.

In [ ]:
# Infrared line sensor inspection with visual display feedback
from microbit import *
from maqueen import Maqueen
import utime

# Initialise robot
robot = Maqueen()

# Test for 30 sample cycles
for _ in range(30):
    sensor_left  = robot.line_left()   # pin13: 1 (dark line) or 0 (light)
    sensor_right = robot.line_right()  # pin14: 1 (dark line) or 0 (light)

    # Visual representation on 5x5 matrix
    if sensor_left == 1 and sensor_right == 1:
        display.show(Image.ARROW_N)  # both on track -> forward arrow
    elif sensor_left == 1 and sensor_right == 0:
        display.show(Image.ARROW_W)  # left on track -> west arrow
    elif sensor_left == 0 and sensor_right == 1:
        display.show(Image.ARROW_E)  # right on track -> east arrow
    else:
        display.show(Image.NO)       # neither on track -> cross icon

    utime.sleep(0.1)

display.clear()

## 9. Practical Robotics Projects {#sec-projects}

In this section, we apply the **lists**, **tuples**, and **dictionaries** from Session 7–8 to design clean, data-driven robotics architectures without relying on untethered print statements.

### Project A — State Machine Obstacle Avoidance Using a Configuration Dictionary

Instead of scattering motor commands, LED calls, and display icons across multiple nested branches, we define a **State Dictionary** `STATE_CONFIG`.

Each state key (such as `"CRUISE"`, `"STOP"`, `"REVERSE"`, and `"TURN_RIGHT"`) maps to a dictionary containing its exact motor speeds, NeoPixel colour tuple, headlight indicator states, and 5x5 LED display icon.

In [ ]:
# Project A: State machine obstacle avoidance with a configuration dictionary
from microbit import *
from maqueen import Maqueen
import utime

robot = Maqueen()

# State behaviour configuration dictionary
STATE_CONFIG = {
    "CRUISE": {
        "motor": (100, 100),       # both motors forward
        "rgb": (0, 255, 0),        # green underglow: safe
        "indicators": (0, 0),      # headlights off
        "icon": Image.HAPPY,       # 5x5 smiling face
    },
    "STOP": {
        "motor": (0, 0),           # motors stopped
        "rgb": (255, 0, 0),        # red underglow: obstacle detected
        "indicators": (1, 1),      # hazard headlights on
        "icon": Image.SKULL,       # 5x5 warning icon
    },
    "REVERSE": {
        "motor": (-80, -80),       # gentle backing up
        "rgb": (255, 100, 0),      # amber underglow
        "indicators": (1, 1),      # headlights on
        "icon": Image.ARROW_S,     # South arrow
    },
    "TURN_RIGHT": {
        "motor": (100, -100),      # spin right
        "rgb": (0, 0, 255),        # blue underglow
        "indicators": (0, 1),      # right headlight on
        "icon": Image.ARROW_E,     # East arrow
    },
}

def apply_state(state_name: str, duration_seconds: float = 0.0) -> None:
    """Apply motor, LED, NeoPixel, and display settings from the state dictionary.

    Args:
        state_name: Key in STATE_CONFIG (e.g. 'CRUISE', 'STOP').
        duration_seconds: Time to hold this state in seconds (default 0.0 for continuous).
    """
    cfg = STATE_CONFIG[state_name]

    # 1. Update 5x5 screen icon
    display.show(cfg["icon"])

    # 2. Update headlights
    robot.led_left(cfg["indicators"][0])
    robot.led_right(cfg["indicators"][1])

    # 3. Update NeoPixel underglow
    r, g, b = cfg["rgb"]
    robot.rgb_front_left(r, g, b)
    robot.rgb_front_right(r, g, b)

    # 4. Update drive motors
    robot.set_motor(0, cfg["motor"][0])
    robot.set_motor(1, cfg["motor"][1])

    if duration_seconds > 0:
        utime.sleep(duration_seconds)


SAFE_DISTANCE_CM = 15.0

# Run 50 obstacle avoidance cycles
for _ in range(50):
    current_dist = robot.read_distance()

    if current_dist > SAFE_DISTANCE_CM or current_dist <= 0:
        # Clear pathway
        apply_state("CRUISE")
    else:
        # Obstacle detected: execute escape manoeuvre sequence
        apply_state("STOP", 0.2)
        apply_state("REVERSE", 0.4)
        apply_state("TURN_RIGHT", 0.5)

    utime.sleep(0.05)

# Clean shutdown
robot.motor_stop_all()
display.clear()
robot.led_left(0)
robot.led_right(0)
robot.rgb_front_left(0, 0, 0)
robot.rgb_front_right(0, 0, 0)

### Project B — Closed-Loop Line Following with a Lookup Dictionary

Instead of multiple `if`/`elif` conditions, we use a **dictionary lookup table** where the key is a 2-element tuple `(left_sensor, right_sensor)` representing the infrared sensor readings.

The lookup table returns the corresponding motor speeds, headlight settings, NeoPixel colour, and display icon in a single operation: `LINE_LOOKUP.get(sensor_key)`.

In [ ]:
# Project B: Line tracking controller using a dictionary lookup table
from microbit import *
from maqueen import Maqueen
import utime

robot = Maqueen()

# Lookup dictionary: maps (left_sensor, right_sensor) tuple to action dictionary
LINE_LOOKUP = {
    (1, 1): {
        "motor": (80, 80),        # both wheels forward
        "icon": Image.ARROW_N,    # straight ahead
        "indicators": (1, 1),     # both headlights on
        "rgb": (0, 255, 0),       # green underglow
    },
    (1, 0): {
        "motor": (0, 80),         # pivot left (drive right wheel)
        "icon": Image.ARROW_W,    # turning left
        "indicators": (1, 0),     # left headlight on
        "rgb": (255, 200, 0),     # amber underglow
    },
    (0, 1): {
        "motor": (80, 0),         # pivot right (drive left wheel)
        "icon": Image.ARROW_E,    # turning right
        "indicators": (0, 1),     # right headlight on
        "rgb": (255, 200, 0),     # amber underglow
    },
    (0, 0): {
        "motor": (0, 0),          # stop (off track)
        "icon": Image.NO,         # cross icon
        "indicators": (0, 0),     # headlights off
        "rgb": (255, 0, 0),       # red warning underglow
    },
}

# Run line-following control loop for 100 iterations
for _ in range(100):
    # Read underside infrared sensors as a tuple key
    sensor_key = (robot.line_left(), robot.line_right())

    # Retrieve action dictionary from lookup table
    action = LINE_LOOKUP.get(sensor_key, LINE_LOOKUP[(0, 0)])

    # Apply 5x5 display icon
    display.show(action["icon"])

    # Apply headlights
    robot.led_left(action["indicators"][0])
    robot.led_right(action["indicators"][1])

    # Apply NeoPixel underglow colour
    r, g, b = action["rgb"]
    robot.rgb_front_left(r, g, b)
    robot.rgb_front_right(r, g, b)

    # Apply motor drive speeds
    robot.set_motor(0, action["motor"][0])
    robot.set_motor(1, action["motor"][1])

    utime.sleep(0.02)  # fast 20 ms control period

# Safe stop
robot.motor_stop_all()
display.clear()
robot.led_left(0)
robot.led_right(0)
robot.rgb_front_left(0, 0, 0)
robot.rgb_front_right(0, 0, 0)

### Project C — Autonomous Mission Planning with a List of Step Dictionaries

In autonomous robotics, navigational routes are often represented as a **sequence of waypoint commands**.

In this project, we create `MISSION_ROUTE` as a **list of dictionaries**. Each dictionary encapsulates the full specification for one leg of the journey: motor speeds, duration, headlight states, NeoPixel colour, display icon, and an audio pitch cue for step transitions.

In [ ]:
# Project C: Mission route dispatcher using a list of step dictionaries
from microbit import *
from maqueen import Maqueen
import utime
import music

robot = Maqueen()

# Route plan: list of command dictionaries
MISSION_ROUTE = [
    {
        "step": 1,
        "motor": (90, 90),         # forward
        "duration": 1.5,
        "indicators": (1, 1),
        "rgb": (0, 255, 0),        # green: cruising
        "icon": Image.ARROW_N,
        "pitch": 523,              # Note C5
    },
    {
        "step": 2,
        "motor": (90, 0),          # pivot turn right
        "duration": 0.65,
        "indicators": (0, 1),      # right headlight
        "rgb": (255, 200, 0),      # amber: turning
        "icon": Image.ARROW_E,
        "pitch": 659,              # Note E5
    },
    {
        "step": 3,
        "motor": (90, 90),         # forward
        "duration": 1.5,
        "indicators": (1, 1),
        "rgb": (0, 255, 0),
        "icon": Image.ARROW_N,
        "pitch": 523,
    },
    {
        "step": 4,
        "motor": (0, 90),          # pivot turn left
        "duration": 0.65,
        "indicators": (1, 0),      # left headlight
        "rgb": (255, 200, 0),
        "icon": Image.ARROW_W,
        "pitch": 587,              # Note D5
    },
    {
        "step": 5,
        "motor": (-80, -80),       # reverse park
        "duration": 1.0,
        "indicators": (1, 1),
        "rgb": (255, 0, 0),        # red: reverse
        "icon": Image.ARROW_S,
        "pitch": 440,              # Note A4
    },
]

def execute_mission(route: list) -> None:
    """Execute a sequence of robotic steps defined in a list of dictionaries.

    Args:
        route: A list of dictionary commands specifying motor, lighting, and sound actions.
    """
    for cmd in route:
        # 1. Display step icon on micro:bit 5x5 grid
        display.show(cmd["icon"])

        # 2. Sound audio tone for step transition
        music.pitch(cmd["pitch"], 150)

        # 3. Set indicator headlights
        robot.led_left(cmd["indicators"][0])
        robot.led_right(cmd["indicators"][1])

        # 4. Set underglow NeoPixels
        r, g, b = cmd["rgb"]
        robot.rgb_front_left(r, g, b)
        robot.rgb_front_right(r, g, b)

        # 5. Drive motors for specified duration
        robot.set_motor(0, cmd["motor"][0])
        robot.set_motor(1, cmd["motor"][1])
        utime.sleep(cmd["duration"])

        # Pause briefly between steps
        robot.motor_stop_all()
        utime.sleep(0.2)

    # Mission complete feedback: display tick and play celebration melody
    robot.motor_stop_all()
    robot.led_left(0)
    robot.led_right(0)
    robot.rgb_front_left(0, 0, 0)
    robot.rgb_front_right(0, 0, 0)
    display.show(Image.YES)
    music.play(music.POWER_UP)


# Run the mission sequence
execute_mission(MISSION_ROUTE)

## 10. Exercises {#sec-exercises}

Apply your understanding of Maqueen robotics programming, lists, and dictionaries to complete the following four challenges.

### Exercise 1: Traffic Light Sequence with a List of Dictionaries {.unnumbered}

Create a list of 3 dictionaries representing an Australian traffic light sequence:
1. **Red**: `(255, 0, 0)`, duration 2.0s, icon `Image.NO`.
2. **Amber**: `(255, 100, 0)`, duration 1.0s, icon `Image.DIAMOND_SMALL`.
3. **Green**: `(0, 255, 0)`, duration 2.0s, icon `Image.YES`.

Iterate through the list 3 times, updating `rgb_front_left` and the 5x5 screen on each step.

### Exercise 2: Automated Reverse Parking Mission {.unnumbered}

Define a list `PARK_MISSION` containing 3 step dictionaries:
1. **Drive forward** for 1.5s (speed 90, green NeoPixels, North arrow).
2. **Stop & inspect** for 0.5s (speed 0, headlights on, Small Diamond icon).
3. **Reverse bay park** for 1.0s (speed -90, red rear NeoPixels, South arrow).

Write a function `run_parking(steps: list) -> None` that iterates through the list to execute the sequence.

### Exercise 3: Wall-Following Escape Routine with State Dictionary {.unnumbered}

Modify the obstacle avoidance controller using a state dictionary so that whenever an obstacle is detected within 20 cm:
1. The robot applies `"BACKUP"` (reverse speed -80, amber NeoPixels) for 0.4 seconds.
2. The robot applies `"SPIN_LEFT"` (left speed -100, right speed 100, West arrow, left headlight) for 0.5 seconds.
3. Test whether the robot can circumnavigate the outer perimeter of a box or barrier.

### Exercise 4 (Extension): Autonomous Patrol with Audio Alert {.unnumbered}

Combine motors, ultrasonic sensing, and the micro:bit `music` module:
1. Drive forward while distance is greater than 20 cm with a smiling face on the display.
2. When an obstacle is detected closer than 20 cm:
   - Stop motors immediately and show `Image.SKULL`.
   - Play a warning tone using `music.pitch(880, 200)` (880 Hz tone for 200 ms).
   - Flash both indicator headlights twice.
   - Reverse and turn away before continuing.

### Reference Solutions {.unnumbered}

Review the reference solutions below after attempting the exercises.

In [ ]:
# Exercise 1 Solution: Traffic Light Sequence with a List of Dictionaries
from microbit import *
from maqueen import Maqueen
import utime

robot = Maqueen()

# List of signal stage dictionaries
TRAFFIC_STAGES = [
    {"name": "Red",   "rgb": (255, 0, 0),   "duration": 2.0, "icon": Image.NO},
    {"name": "Amber", "rgb": (255, 100, 0), "duration": 1.0, "icon": Image.DIAMOND_SMALL},
    {"name": "Green", "rgb": (0, 255, 0),   "duration": 2.0, "icon": Image.YES},
]

# Run 3 complete cycles
for _ in range(3):
    for stage in TRAFFIC_STAGES:
        display.show(stage["icon"])
        r, g, b = stage["rgb"]
        robot.rgb_front_left(r, g, b)  # set front-left NeoPixel
        utime.sleep(stage["duration"])

# Turn off NeoPixel and clear display
robot.rgb_front_left(0, 0, 0)
display.clear()

In [ ]:
# Exercise 2 Solution: Automated Reverse Parking Mission
from microbit import *
from maqueen import Maqueen
import utime

robot = Maqueen()

# List of parking steps
PARK_MISSION = [
    {
        "motor": (90, 90),
        "duration": 1.5,
        "rgb_front": (0, 255, 0),
        "rgb_rear": (0, 0, 0),
        "indicators": (0, 0),
        "icon": Image.ARROW_N,
    },
    {
        "motor": (0, 0),
        "duration": 0.5,
        "rgb_front": (0, 0, 0),
        "rgb_rear": (0, 0, 0),
        "indicators": (1, 1),
        "icon": Image.DIAMOND_SMALL,
    },
    {
        "motor": (-90, -90),
        "duration": 1.0,
        "rgb_front": (0, 0, 0),
        "rgb_rear": (255, 0, 0),
        "indicators": (0, 0),
        "icon": Image.ARROW_S,
    },
]

def run_parking(steps: list) -> None:
    """Execute parking steps from a list of configuration dictionaries.

    Args:
        steps: List of step dictionaries.
    """
    for step in steps:
        display.show(step["icon"])
        robot.led_left(step["indicators"][0])
        robot.led_right(step["indicators"][1])

        rf, gf, bf = step["rgb_front"]
        robot.rgb_front_left(rf, gf, bf)
        robot.rgb_front_right(rf, gf, bf)

        rr, gr, br = step["rgb_rear"]
        robot.rgb_rear_left(rr, gr, br)
        robot.rgb_rear_right(rr, gr, br)

        robot.set_motor(0, step["motor"][0])
        robot.set_motor(1, step["motor"][1])
        utime.sleep(step["duration"])

    robot.motor_stop_all()
    robot.led_left(0)
    robot.led_right(0)
    robot.rgb_front_left(0, 0, 0)
    robot.rgb_front_right(0, 0, 0)
    robot.rgb_rear_left(0, 0, 0)
    robot.rgb_rear_right(0, 0, 0)
    display.show(Image.YES)


run_parking(PARK_MISSION)

In [ ]:
# Exercise 3 Solution: Wall-Following Escape Routine (Left-Turn Rule)
from microbit import *
from maqueen import Maqueen
import utime

robot = Maqueen()

ESCAPE_STATES = {
    "FORWARD":   {"motor": (100, 100),  "icon": Image.ARROW_N, "rgb": (0, 255, 0), "led": (0, 0)},
    "BACKUP":    {"motor": (-80, -80),  "icon": Image.ARROW_S, "rgb": (255, 100, 0), "led": (1, 1)},
    "SPIN_LEFT": {"motor": (-100, 100), "icon": Image.ARROW_W, "rgb": (0, 0, 255), "led": (1, 0)},
}

def set_escape_state(name: str, duration: float = 0.0) -> None:
    st = ESCAPE_STATES[name]
    display.show(st["icon"])
    robot.led_left(st["led"][0])
    robot.led_right(st["led"][1])
    r, g, b = st["rgb"]
    robot.rgb_front_left(r, g, b)
    robot.rgb_front_right(r, g, b)
    robot.set_motor(0, st["motor"][0])
    robot.set_motor(1, st["motor"][1])
    if duration > 0:
        utime.sleep(duration)


CLEARANCE_LIMIT_CM = 20.0

for _ in range(30):
    dist_cm = robot.read_distance()

    if dist_cm > CLEARANCE_LIMIT_CM or dist_cm <= 0:
        set_escape_state("FORWARD")
    else:
        # Obstacle detected: stop, backup, and spin left
        robot.motor_stop_all()
        utime.sleep(0.2)
        set_escape_state("BACKUP", 0.4)
        set_escape_state("SPIN_LEFT", 0.5)

    utime.sleep(0.05)

robot.motor_stop_all()
display.clear()
robot.led_left(0)
robot.led_right(0)
robot.rgb_front_left(0, 0, 0)
robot.rgb_front_right(0, 0, 0)

In [ ]:
# Exercise 4 Solution: Autonomous Patrol with Audio & Screen Alert
from microbit import *
from maqueen import Maqueen
import utime
import music

robot = Maqueen()

SPEED_CRUISE   = 90
SAFE_DISTANCE  = 20.0
ALERT_PITCH_HZ = 880
ALERT_MS       = 200

for _ in range(25):
    distance = robot.read_distance()

    if distance > SAFE_DISTANCE or distance <= 0:
        display.show(Image.HAPPY)
        robot.rgb_front_left(0, 255, 0)
        robot.rgb_front_right(0, 255, 0)
        robot.set_motor(0, SPEED_CRUISE)
        robot.set_motor(1, SPEED_CRUISE)
    else:
        # Halt and trigger audiovisual alert
        robot.motor_stop_all()
        display.show(Image.SKULL)
        robot.rgb_front_left(255, 0, 0)
        robot.rgb_front_right(255, 0, 0)

        # Emit pitch cue through speaker
        music.pitch(ALERT_PITCH_HZ, ALERT_MS)

        # Flash headlights twice
        for _ in range(2):
            robot.led_left(1)
            robot.led_right(1)
            utime.sleep(0.1)
            robot.led_left(0)
            robot.led_right(0)
            utime.sleep(0.1)

        # Reverse and turn away
        display.show(Image.ARROW_S)
        robot.set_motor(0, -80)
        robot.set_motor(1, -80)
        utime.sleep(0.4)

        display.show(Image.ARROW_E)
        robot.set_motor(0, 100)
        robot.set_motor(1, -100)
        utime.sleep(0.5)

        robot.motor_stop_all()

    utime.sleep(0.05)

robot.motor_stop_all()
display.clear()
robot.led_left(0)
robot.led_right(0)
robot.rgb_front_left(0, 0, 0)
robot.rgb_front_right(0, 0, 0)

## 11. Useful Resources {#sec-resources}

The following resources provide additional references and documentation for programming the BBC micro:bit and the DFRobot Micro:Maqueen robot platform:

- [micro:bit Python Editor (v3)](https://python.microbit.org/v/3) — Official browser-based editor and simulator.
- [micro:bit MicroPython Documentation](https://microbit-micropython.readthedocs.io) — Complete API documentation for on-board display, buttons, accelerometer, and radio.
- [DFRobot Micro:Maqueen Wiki](https://wiki.dfrobot.com/micro_Maqueen_for_micro_bit_SKU_ROB0148) — Hardware schematics, motor specifications, and sensor details.
- [DFRobot Official Product Page](https://www.dfrobot.com/product-1783.html) — Product specifications and driver resources.